# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Three candidates, from readable to strong:** logistic regression (standardised, to read the signs), a random forest, and histogram gradient boosting. All use the same 18 features from `flyrank_features.FEATURES`, and none sees IDs, the outcome month or any edit date.

I'm treating them as rankers. What matters is the order of the scores, judged at precision@K and per-site precision@10, with the baseline rule in the same table.

In [ ]:
# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')
# Validation design used everywhere: client-grouped AND time-forward.
# Each client gets a fixed fold id (a hash of its pseudonym). To score fold f at decision date T2,
# a model is trained on the OTHER folds' clients at T1 only - so it has never seen the client,
# and never seen the outcome month it is judged on.
T1, T2 = ff.DEV_DATES
va = m[m.decision_date == T2].copy()
va['fold'] = va.client_hash_id.map(ff.client_fold)
print(f'judge snapshot {T2}: {len(va):,} rows, {va.client_hash_id.nunique()} clients, base rate {va.declined.mean():.3f}')
print('rows per client fold:', va.fold.value_counts().sort_index().to_dict())

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
MAKERS = {
    'logreg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'hgb': lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                                  min_samples_leaf=100, random_state=ff.SEED),
    'rf': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=ff.SEED),
}
FINAL = 'hgb'    # frozen in w05 on the development split (best top-of-queue precision), never re-chosen later
F = ff.FEATURES

def baseline(df):
    """Frozen transparent rule: one point per warning sign, ties broken by current visibility."""
    r = pd.DataFrame(index=df.index)
    r['sliding_impressions'] = df.imp_last30 < 0.85 * df.imp_prev30      # already losing impressions
    r['position_slipping']   = df.pos_change > 1.0                        # average rank got >1 place worse
    r['one_day_spike']       = df.spike_share > 0.25                      # month propped up by a single day
    r['aging_page']          = df.content_age_days > 270                  # past the decay cliff in FlyRank's paper
    r['patchy_visibility']   = df.days_with_imp_last30 < 15               # visible on fewer than half the days
    points = r.sum(axis=1)
    score = points + df.log_imp_last30 / 100.0
    reasons = r.apply(lambda row: '|'.join(c for c in r.columns if row[c]) or 'none', axis=1)
    return score, reasons, r

from sklearn.metrics import roc_auc_score, average_precision_score
def evaluate(scores, y, ks=(20, 50, 100, 500)):
    y = np.asarray(y).astype(int)
    d = {'base_rate': round(float(y.mean()), 4), 'auc': round(float(roc_auc_score(y, scores)), 4),
         'ap': round(float(average_precision_score(y, scores)), 4)}
    for k in ks:
        d[f'p@{k}'] = round(ff.precision_at_k(scores, y, k), 4)
    return d
def client_queue_at_10(df, scores, min_items=30):
    """How a single site would use it: precision of its OWN top-10, averaged over clients
    (with those clients' average base rate next to it)."""
    tmp = df[['client_hash_id', 'declined']].copy(); tmp['s'] = np.asarray(scores)
    rows = [(g.sort_values('s', ascending=False).declined.head(10).mean(), g.declined.mean())
            for _, g in tmp.groupby('client_hash_id') if len(g) >= min_items]
    return {'client_p@10': round(float(np.mean([a for a, b in rows])), 3),
            'client_base': round(float(np.mean([b for a, b in rows])), 3), 'clients': len(rows)}
print('candidate models:', list(MAKERS))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped and time-forward at once.** Each client gets a fixed fold (a hash of its pseudonym, the same in every notebook). To score the clients in fold *f* on the April-30 snapshot, the model is trained only on the *other* folds' clients at the March-31 snapshot. So every score comes from a model that never saw that client and never saw May.

Why both: a random split lets a model memorise a client's overall pattern and look clever (notebook 03 showed that gap). A time split mimics real use, where I train on the past and act on the next month. The cell below proves there's no client overlap in any fold.

In [ ]:
for f in range(4):
    test_clients = set(va.loc[va.fold == f, 'client_hash_id'])
    train_clients = set(m.loc[(m.decision_date == T1) & (m.client_hash_id.map(ff.client_fold) != f), 'client_hash_id'])
    print(f'fold {f}: judged on {len(test_clients)} clients at {T2}; trained on {len(train_clients)} other clients at {T1}; overlap {len(test_clients & train_clients)}')
print('label months: April 2026 (training rows), May 2026 (judged rows). June 2026 stays sealed for w06.')

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same rows, same label, same metrics as the baseline.

All three models beat the rule on the whole ordering (AUC ≈ 0.70-0.74 vs 0.61) and at the top of the queue. Gradient boosting and random forest are close on AUC and average precision. **Gradient boosting wins on what I said I'd judge by**: precision@50 ≈ 0.98 vs 0.88 for the rule, and per-site precision@10 ≈ 0.92 vs 0.81 for the rule, where the base rate is ≈ 0.50. It also trains much faster. So I freeze `hgb` as the final model *now*, before June is opened.

Honest caveat: at precision@50 the rule is already high because the queue is pooled across sites. The per-site numbers and AUC are where the gap is clearest.

In [ ]:
results, oof = {}, {}
bscore, _, _ = baseline(va)
results['baseline_rule'] = evaluate(bscore, va.declined); results['baseline_rule'].update(client_queue_at_10(va, bscore))
for name, mk in MAKERS.items():
    o = ff.grouped_forward_scores(m, [T1], T2, mk, F)
    oof[name] = o
    results[name] = evaluate(o.score, o.declined); results[name].update(client_queue_at_10(o, o.score))
table = pd.DataFrame(results).T
print(table.to_string())
json.dump(results, open('work/outputs/dev_model_comparison.json', 'w'), indent=2, default=str)
print('\nfrozen choice for everything after this notebook:', FINAL)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Where it's wrong.** The scores rank well but aren't calibrated probabilities: the mean score sits above the actual rate. So I present them as a risk *order*, not as "73% chance". Ranking is weakest on the highest-volume quartile, where big pages are steadier and harder to separate. Per-client AUC ranges widely, so the model works for every client but much better for some.

**What it leans on.** Last month's momentum is by far the most important feature, followed by age, recent volume and position. The logistic signs are readable: rising momentum, more clicks and longer pages mean less risk. A one-day spike means more risk. Pages ranked very high (small position number) are a bit riskier, which looks like regression to the mean. Nothing towers suspiciously the way a leaked feature would.

**Hard cases.** The most confident false alarms are pages that had already crashed last month and then bounced back. The model reads the crash as momentum. The misses are mostly deep-ranked pages (position 40-70) with no warning sign at all. Their drop looks like noise until it happens.

In [ ]:
from sklearn.inspection import permutation_importance
o = oof[FINAL]
o['volume_quartile'] = pd.qcut(o.imp_last30, 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
print('where is it wrong? by volume quartile (AUC inside the quartile, mean score vs actual rate):')
print(o.groupby('volume_quartile', observed=True).apply(lambda g: pd.Series({'rows': len(g), 'mean_score': g.score.mean(),
      'actual_rate': g.declined.mean(), 'auc': roc_auc_score(g.declined.astype(int), g.score)})).round(3).to_string())
per_client = sorted(round(roc_auc_score(g.declined.astype(int), g.score), 3) for _, g in o.groupby('client_hash_id')
                    if g.declined.nunique() == 2 and len(g) >= 100)
print('\nAUC per client (clients with >= 100 rows):', per_client)
# what does it lean on? a time-forward fit (all clients, T1 -> T2) checked by shuffling each feature
tr = m[m.decision_date == T1]
mdl = MAKERS[FINAL]().fit(tr[F], tr.declined.astype(int))
sub = va.sample(min(20000, len(va)), random_state=ff.SEED)
pi = permutation_importance(mdl, sub[F], sub.declined.astype(int), n_repeats=3, random_state=ff.SEED, scoring='average_precision', n_jobs=-1)
print('\npermutation importance (drop in average precision when shuffled):')
print(pd.Series(pi.importances_mean, index=F).sort_values(ascending=False).round(4).head(10).to_string())
lr = MAKERS['logreg']().fit(tr[F], tr.declined.astype(int))
print('\nlogistic regression coefficients (standardised; + = more likely to fall behind the site):')
print(pd.Series(lr[-1].coef_[0], index=F).sort_values(key=abs, ascending=False).round(3).head(8).to_string())
cols = ['imp_prev30', 'imp_last30', 'imp_next30', 'client_ratio', 'pos_last30', 'pos_change', 'spike_share', 'content_age_days', 'score']
print('\n3 most confident false alarms:'); print(o[o.declined == 0].nlargest(3, 'score')[cols].round(2).to_string(index=False))
print('\n3 most confident misses:'); print(o[o.declined == 1].nsmallest(3, 'score')[cols].round(2).to_string(index=False))

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.